# OpenCV 机器学习预处理与特征提取教程

本笔记本介绍了 OpenCV 的基础知识，以及与机器学习流水线相关的特定技术，例如特征提取和图像分割。

## 1. 环境设置 (Setup)

我们需要安装 `opencv-python`, `matplotlib`, 和 `numpy` 库。

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

print(f"OpenCV 版本: {cv2.__version__}")

## 2. 基础图像操作 (Basic Image Operations)

加载和显示图像是任何计算机视觉 (CV) 任务的第一步。

In [ ]:
# 加载图像
image_path = '../images/pipeline.png'
img_bgr = cv2.imread(image_path)

if img_bgr is None:
    print(f"错误: 无法在 {image_path} 找到图像")
else:
    # 将 BGR 转换为 RGB 以便使用 matplotlib 显示
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=(10, 6))
    plt.imshow(img_rgb)
    plt.title(f'原始图像 {img_rgb.shape}')
    plt.axis('off')
    plt.show()

## 3. 面向机器学习的预处理 (Pre-processing for ML)

CV 中的数据清洗通常涉及调整大小 (标准化) 和降噪。

In [ ]:
if img_bgr is not None:
    # 1. 调整大小 (Resize): 标准化输入尺寸 (例如，为了满足神经网络的输入要求)
    target_size = (256, 256)
    img_resized = cv2.resize(img_rgb, target_size)
    
    # 2. 灰度转换 (Grayscale): 降维 (从 3 通道变为 1 通道)
    img_gray = cv2.cvtColor(img_resized, cv2.COLOR_RGB2GRAY)
    
    # 3. 模糊 (Blurring): 降噪 (使用高斯模糊)
    img_blurred = cv2.GaussianBlur(img_gray, (5, 5), 0)
    
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    ax[0].imshow(img_resized); ax[0].set_title('调整大小后 (256x256)')
    ax[1].imshow(img_gray, cmap='gray'); ax[1].set_title('灰度图')
    ax[2].imshow(img_blurred, cmap='gray'); ax[2].set_title('模糊后 (降噪)')
    plt.show()

## 4. 特征提取 (Feature Extraction)

将原始像素转换为有意义的特征对于传统机器学习算法 (如 SVM, 随机森林等) 至关重要。

### A. 颜色直方图 (Color Histograms)
描述像素强度的分布。对于基于颜色的图像分类非常有用。

In [ ]:
if img_bgr is not None:
    colors = ('b', 'g', 'r')
    plt.figure(figsize=(10, 5))
    plt.title('颜色直方图')
    plt.xlabel('箱 (Bins)')
    plt.ylabel('像素数量')
    
    for i, col in enumerate(colors):
        # 计算直方图: 图像, 通道, 掩膜, 直方图尺寸, 范围
        hist = cv2.calcHist([img_bgr], [i], None, [256], [0, 256])
        plt.plot(hist, color=col)
        plt.xlim([0, 256])
        
    plt.show()

### B. 边缘特征 (Edge Features - Canny)
捕捉图像的结构信息。

In [ ]:
if img_bgr is not None:
    # 使用模糊后的灰度图像以获得更好的边缘检测效果
    edges = cv2.Canny(img_blurred, 50, 150)
    
    plt.figure(figsize=(8, 8))
    plt.imshow(edges, cmap='gray')
    plt.title('Canny 边缘检测图')
    plt.axis('off')
    plt.show()

### C. ORB (Oriented FAST and Rotated BRIEF)
SIFT/SURF 的一种快速高效的替代方案。它用于检测关键点 (兴趣点) 并计算描述符 (描述这些点的向量)。

In [ ]:
if img_bgr is not None:
    # 初始化 ORB 检测器
    orb = cv2.ORB_create()
    
    # 使用 ORB 寻找关键点和描述符
    kp, des = orb.detectAndCompute(img_gray, None)
    
    # 仅绘制关键点位置，不绘制大小和方向
    img_orb = cv2.drawKeypoints(img_gray, kp, None, color=(0, 255, 0), flags=0)
    
    plt.figure(figsize=(10, 10))
    plt.imshow(img_orb)
    plt.title(f'检测到的 ORB 关键点数量: {len(kp)}')
    plt.axis('off')
    plt.show()
    
    print(f"描述符形状: {des.shape if des is not None else 'None'}")
    print("每个关键点由一个 32 维向量描述 (二进制特征)。")

## 5. 无监督学习: 图像分割 (Unsupervised Learning: Image Segmentation)

### K-Means 聚类
我们可以使用 K-Means 对像素颜色进行聚类。这可以有效地将图像分割成 $K$ 种主色调。

In [ ]:
if img_bgr is not None:
    # 将图像重塑为像素的 2D 数组 (高度 * 宽度, 3)
    pixel_values = img_rgb.reshape((-1, 3))
    # 转换为浮点数
    pixel_values = np.float32(pixel_values)
    
    # 定义停止准则 = ( 类型, 最大迭代次数, 精度epsilon )
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 0.2)
    k = 4 # 聚类数量 (颜色数)
    
    _, labels, (centers) = cv2.kmeans(pixel_values, k, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)
    
    # 转换回 8 位数值
    centers = np.uint8(centers)
    
    # 将标签映射回中心点颜色
    segmented_data = centers[labels.flatten()]
    
    # 重塑回原始图像尺寸
    segmented_image = segmented_data.reshape(img_rgb.shape)
    
    plt.figure(figsize=(10, 6))
    plt.imshow(segmented_image)
    plt.title(f'K-Means 分割结果 (K={k})')
    plt.axis('off')
    plt.show()